In [1]:
import yaml
import os
import torch
import numpy as np
import argparse
import time
import pickle
from tqdm import tqdm
import torch
import torch.nn as nn

from models.mobrecon_ds import LargeModel_Extra
from datasets.freihand_ty import Freihand

from torch.utils.data import DataLoader, random_split
from torchvision.transforms import ToTensor

# from torch.utils.tensorboard import SummaryWriter
from torchvision.transforms.functional import to_pil_image

In [2]:
!pip install onnx onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 18.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 26.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [onnxruntime] [onnxruntime]


In [3]:
model = LargeModel_Extra(None)
pth_path = 'pretrain/100.pt'
model.load_state_dict(torch.load(pth_path), strict=False)
model.eval()

LargeModel_Extra(
  (backbone): DenseStack_Backbone_like_prev(
    (pre_layer): Sequential(
      (0): Sequential(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
      )
      (1): mobile_unit(
        (conv3x3): Sequential(
          (0): Sequential(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): ReLU()
          )
        )
        (conv1x1): Sequential(
          (0): Conv2d(32, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU()
        )
      )
    )
    (thrink): Sequential(
      (0): Conv2d(256, 64, kernel_size=(1, 1), stride=(1, 1), bias=F

In [4]:
input_image = torch.ones((1,3,256,256))
b = model(input_image)
b

{'keypoints': tensor([[[0.0638, 0.0585, 0.1075],
          [0.0590, 0.0695, 0.1086],
          [0.0639, 0.0601, 0.1077],
          [0.0664, 0.0566, 0.1120],
          [0.0609, 0.0635, 0.1120],
          [0.0611, 0.0605, 0.1040],
          [0.0575, 0.0753, 0.1053],
          [0.0558, 0.0738, 0.1062],
          [0.0548, 0.0742, 0.1058],
          [0.0590, 0.0689, 0.1096],
          [0.0611, 0.0658, 0.1098],
          [0.0557, 0.0742, 0.1060],
          [0.0585, 0.0716, 0.1077],
          [0.0538, 0.0770, 0.1062],
          [0.0582, 0.0723, 0.1076],
          [0.0559, 0.0770, 0.1049],
          [0.0649, 0.0589, 0.1094],
          [0.0572, 0.0716, 0.1081],
          [0.0603, 0.0669, 0.1095],
          [0.0575, 0.0740, 0.1030],
          [0.0585, 0.0693, 0.1100]]], grad_fn=<ViewBackward0>)}

In [6]:
input_image = torch.zeros((1,3,256,256))
torch.onnx.export(model, input_image, "pretrain/100.onnx", export_params=True, verbose=False, input_names=['input0'], output_names=['output0'])